<span style="font-family: 'Courier New', monospace;">

*AI-generated draft (Claude, Anthropic) — for review. All parameters and figures are derived from version-controlled scripts and data.*

# 25 — Scene 1 / not-Scene 1 sorter

Click through the contact sheets produced by `scripts/scene_sampler.py` and file each recording into **Scene 1** (the front-on Mushroom view used for scale-worm counts) or **Not Scene 1**.

**How to use**

1. Set `SESSION_DIR` below to the sampler output you want to sort (e.g. a validation month, or the full-run directory).
2. Run all cells. A contact sheet appears with three buttons.
3. **Scene 1** / **Not Scene 1** move the sheet into `scene1/` or `not_scene1/` and advance. **Skip** leaves it in place and advances (revisit it next session).
4. Every click is appended to `sort_log.csv` in the session directory. Sorting is resumable — already-filed sheets are gone from `contact_sheets/`, so re-running picks up where you left off.

The move/log logic lives in `scripts/scene_sorter.py` (unit-tested); this notebook is only the UI.

</span>

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import ipywidgets as w
import matplotlib.pyplot as plt
from IPython.display import display

import scene_sorter as sorter

# --- CONFIG: point this at the session you want to sort ---------------------
SESSION_DIR = Path.cwd().parent / "scene_sorting" / "validation_2023_03"

print("Sorting:", SESSION_DIR)
print("Status :", sorter.counts(SESSION_DIR))

In [ ]:
skipped: set[str] = set()  # sheet filenames skipped this session

img_out = w.Output()
status = w.HTML()


def _remaining():
    return [p for p in sorter.pending_sheets(SESSION_DIR) if p.name not in skipped]


def _render():
    img_out.clear_output(wait=True)
    rem = _remaining()
    c = sorter.counts(SESSION_DIR)
    status.value = (
        f"<b>Scene 1:</b> {c['scene1']} &nbsp;&nbsp; "
        f"<b>Not Scene 1:</b> {c['not_scene1']} &nbsp;&nbsp; "
        f"<b>Remaining:</b> {len(rem)} "
        f"(skipped this session: {len(skipped)})"
    )
    with img_out:
        if not rem:
            print("\u2705 Nothing left to sort in this session.")
            return
        sheet = rem[0]
        fig, ax = plt.subplots(figsize=(15, 8))
        ax.imshow(plt.imread(sheet))
        ax.axis("off")
        ax.set_title(sheet.stem, fontsize=11)
        plt.show()


def _decide(decision):
    rem = _remaining()
    if not rem:
        _render()
        return
    sheet = rem[0]
    if decision == "skip":
        skipped.add(sheet.name)
    sorter.apply_decision(SESSION_DIR, sheet, decision)
    _render()


b_s1 = w.Button(description="Scene 1", button_style="success", icon="check")
b_no = w.Button(description="Not Scene 1", button_style="danger", icon="times")
b_sk = w.Button(description="Skip", button_style="warning", icon="forward")
b_s1.on_click(lambda _: _decide("scene1"))
b_no.on_click(lambda _: _decide("not_scene1"))
b_sk.on_click(lambda _: _decide("skip"))

display(w.VBox([status, w.HBox([b_s1, b_no, b_sk]), img_out]))
_render()